# 08 — vLLM: RULER Benchmark

This notebook runs the same RULER benchmark as notebook 07, but using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B and
KV cache compression via vLLM's built-in compression support.

We test two compression algorithms at multiple ratios:
- **full_replacement** — full KV cache replacement after prefill
- **filtering** — token filtering during decoding

Scoring uses `calculate_metrics` from the kvpress evaluation framework
(same scoring as the [kvpress leaderboard](https://huggingface.co/spaces/nvidia/kvpress-leaderboard))
for apples-to-apples comparison with notebook 07.

Results are saved to `results/vllm_ruler/` for comparison in later notebooks.

## Configuration

In [1]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

RULER_DATA_DIRS = ["4096", "8192"]

FRACTION = 0.01

SEED = 42

MAX_NEW_TOKENS = 128

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [2]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


Using FORK vLLM (version: dev)
Using FORK vLLM (version: dev)


In [4]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n\u26a0  GPU memory is not free \u2014 a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

def get_gpu_memory_used_gb() -> float:
    """Actual GPU memory used, measured at the CUDA driver level."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

GPU:        NVIDIA A100-SXM4-40GB
GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)
Reserved:   0.00 GB
Free:       42.4 GB (approx)


In [5]:
KVPRESS_FORK_DIR = "/opt/app-root/src/kvpress-fork"
EVAL_DIR = os.path.join(KVPRESS_FORK_DIR, "evaluation")
sys.path.insert(0, EVAL_DIR)
from benchmarks.ruler.calculate_metrics import calculate_metrics as ruler_calculate_metrics
print(f"RULER scoring from: {EVAL_DIR}")

RULER scoring from: /opt/app-root/src/kvpress-fork/evaluation
RULER scoring from: /opt/app-root/src/kvpress-fork/evaluation


## 1. Load RULER Dataset

In [6]:
from datasets import load_dataset

ruler_datasets = {}
for data_dir in RULER_DATA_DIRS:
    df = load_dataset("simonjegou/ruler", data_dir=data_dir, split="test").to_pandas()
    if FRACTION < 1.0:
        df = df.sample(frac=FRACTION, random_state=SEED)
    df["context_length"] = int(data_dir)
    ruler_datasets[data_dir] = df
    tasks = sorted(df["task"].unique())
    print(f"RULER {data_dir}: {len(df)} examples, {len(tasks)} tasks")
    print(f"  Tasks: {tasks}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RULER 4096: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 4096: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 8192: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']
RULER 8192: 65 examples, 12 tasks
  Tasks: ['cwe', 'fwe', 'niah_multikey_1', 'niah_multikey_2', 'niah_multikey_3', 'niah_multiquery', 'niah_multivalue', 'niah_single_1', 'niah_single_2', 'niah_single_3', 'qa_1', 'qa_2']


## 2. Prepare Prompts

Apply the model's chat template to each RULER example to produce
the final prompt for vLLM batch inference.

In [7]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

ruler_df = pd.concat(ruler_datasets.values(), ignore_index=True)

prompts = []
for _, row in ruler_df.iterrows():
    user_msg = row["context"] + "\n\n" + row["question"]
    messages = [{"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if row["answer_prefix"]:
        prompt += row["answer_prefix"]
    prompts.append(prompt)

ruler_df["prompt"] = prompts
print(f"Prepared {len(ruler_df)} prompts")

Prepared 130 prompts
Prepared 130 prompts


## 3. Run Batch Inference

For each (algorithm, compression_ratio) combination, create a vLLM engine
and process all prompts in batch. The engine is destroyed between configs
to free GPU memory.

In [8]:
import time
import random
import numpy as np
from vllm import LLM, SamplingParams

# Deterministic seeds — matches evaluate.py _setup_deterministic_seeds()
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

prompts = ruler_df["prompt"].tolist()

all_metrics = {}
summary_rows = []
all_predictions = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press_kwargs in configs:
    llm = None
    try:
        label = f"{press_name} | ratio={ratio}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(prompts)} prompts)")
        print(f"{'='*60}")

        if press_kwargs is not None:
            llm = LLM(**press_kwargs)
        else:
            llm = LLM(
                model=MODEL_NAME,
                dtype="auto",
                gpu_memory_utilization=0.90,
                trust_remote_code=True,
                attention_config={"backend": "FLASH_ATTN"},
                enable_prefix_caching=False,
            )

        mem_before = get_gpu_memory_used_gb()
        start = time.perf_counter()

        outputs = llm.generate(prompts, sampling_params)

        batch_elapsed = time.perf_counter() - start
        mem_after = get_gpu_memory_used_gb()
        peak_mem = max(mem_before, mem_after)

        # Build scoring DataFrame with original answer column types from .to_pandas()
        df_eval = ruler_df[["task", "answer", "context_length"]].copy()
        df_eval["predicted_answer"] = [o.outputs[0].text.strip() for o in outputs]

        # Score per context_length — same flow as evaluate.py
        # .copy() ensures calculate_metrics receives an owned DataFrame,
        # matching evaluate.py which passes self.df directly
        for ctx_len, df_ctx in df_eval.groupby("context_length"):
            metrics = ruler_calculate_metrics(df_ctx.copy())
            key = f"{press_name}__{ratio}__{ctx_len}"
            all_metrics[key] = metrics

            avg_score = sum(m["string_match"] for m in metrics.values()) / len(metrics)
            summary_rows.append({
                "press": press_name, "compression_ratio": ratio,
                "context_length": ctx_len, "avg_score": round(avg_score, 2),
                "mean_time": round(batch_elapsed / len(prompts), 3),
                "peak_gpu_mem_gb": round(peak_mem, 3),
            })

        # Collect predictions for saving
        df_preds = df_eval.copy()
        df_preds["framework"] = "vllm"
        df_preds["press"] = press_name
        df_preds["compression_ratio"] = ratio
        df_preds["elapsed_sec"] = round(batch_elapsed / len(prompts), 3)
        df_preds["peak_gpu_mem_gb"] = round(peak_mem, 3)
        all_predictions.append(df_preds)

        total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0

        print(f"  Batch done in {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal configurations: {len(summary_rows)}")


Running: no_press | ratio=0.0 (130 prompts)

Running: no_press | ratio=0.0 (130 prompts)
INFO 08-31 13:24:19 [utils.py:233] non-default args: {'trust_remote_code': True, 'enable_prefix_caching': False, 'disable_log_stats': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.FLASH_ATTN: 'vllm.v1.attention.backends.flash_attn.FlashAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': 'Qwen/Qwen3-8B'}
INFO 08-31 13:24:20 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 08-31 13:24:20 [model.py:1582] Using max model len 40960
INFO 08-31 13:24:20 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-31 13:24:20 [vllm.py:795] Asynchronous sched

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


(EngineCore pid=106148) INFO 08-31 13:24:30 [core.py:103] Initializing a V1 LLM engine (vdev) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_c

(EngineCore pid=106148) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=106148) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:01<00:04,  1.03s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:02<00:03,  1.06s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:03<00:02,  1.08s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:04<00:01,  1.01s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:04<00:00,  1.28it/s]
Loading safetensors checkpoint shards: 100% Complet

(EngineCore pid=106148) INFO 08-31 13:24:38 [default_loader.py:384] Loading weights took 4.52 seconds
(EngineCore pid=106148) INFO 08-31 13:24:39 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 5.540911 seconds
(EngineCore pid=106148) INFO 08-31 13:24:43 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/f1781885f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=106148) INFO 08-31 13:24:43 [backends.py:1048] Dynamo bytecode transform time: 4.21 s
(EngineCore pid=106148) INFO 08-31 13:24:45 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.308 s
(EngineCore pid=106148) INFO 08-31 13:24:45 [monitor.py:48] torch.compile took 5.87 s in total
(EngineCore pid=106148) INFO 08-31 13:24:45 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/b5b87fb51ca98babdcdb46b974078c98a405315eff9cadba9be7225f8

...

(EngineCore pid=108674) INFO 08-31 13:49:56 [default_loader.py:384] Loading weights took 4.49 seconds
(EngineCore pid=108674) INFO 08-31 13:49:57 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 5.470012 seconds
(EngineCore pid=108674) INFO 08-31 13:50:01 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/f1781885f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=108674) INFO 08-31 13:50:01 [backends.py:1048] Dynamo bytecode transform time: 4.21 s
(EngineCore pid=108674) INFO 08-31 13:50:03 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.305 s
(EngineCore pid=108674) INFO 08-31 13:50:03 [monitor.py:48] torch.compile took 5.86 s in total
(EngineCore pid=108674) INFO 08-31 13:50:03 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/b5b87fb51ca98babdcdb46b974078c98a405315eff9cadba9be7225f8

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.97it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 26.61it/s]


(EngineCore pid=108674) INFO 08-31 13:50:10 [gpu_model_runner.py:5807] Graph capturing finished in 4 secs, took 0.52 GiB
(EngineCore pid=108674) INFO 08-31 13:50:10 [gpu_worker.py:617] CUDA graph pool memory: 0.52 GiB (actual), 0.52 GiB (estimated), difference: 0.0 GiB (0.8%).
(EngineCore pid=108674) INFO 08-31 13:50:10 [core.py:281] init engine (profile, create kv cache, warmup model) took 13.37 seconds
(EngineCore pid=108674) INFO 08-31 13:50:11 [vllm.py:795] Asynchronous scheduling is enabled.
INFO 08-31 13:50:11 [llm.py:391] Supported tasks: ['generate']


Processed prompts: 100%|██████████| 130/130 [04:29<00:00,  2.07s/it, est. speed input: 2839.61 toks/s, output: 32.28 toks/s]
[rank0]:[W831 13:54:42.950790935 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


  Batch done in 270.8s — 32.1 tok/s — peak mem=40.38 GB
  Batch done in 270.8s — 32.1 tok/s — peak mem=40.38 GB
(EngineCore pid=108674) INFO 08-31 13:54:42 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=108674) INFO 08-31 13:54:42 [core.py:1224] Shutdown complete

Total configurations: 18

Total configurations: 18


## 4. Results

Scores computed using `calculate_metrics` from the kvpress evaluation
framework — same string-match scoring as the kvpress leaderboard.

In [9]:
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

           press  compression_ratio  context_length  avg_score  mean_time  peak_gpu_mem_gb
        no_press               0.00            4096      93.54      0.548           40.375
        no_press               0.00            8192      90.79      0.548           40.375
full_replacement               0.01            4096      92.92      0.592           40.381
full_replacement               0.01            8192      89.88      0.592           40.381
full_replacement               0.25            4096      91.18      0.562           40.377
full_replacement               0.25            8192      84.91      0.562           40.377
full_replacement               0.50            4096      80.00      0.547           40.377
full_replacement               0.50            8192      73.82      0.547           40.377
full_replacement               0.75            4096      59.86      0.543           40.375
full_replacement               0.75            8192      59.10      0.543           40.375

In [10]:
for key, metrics in sorted(all_metrics.items()):
    print(f"\n{key}")
    for task, scores in sorted(metrics.items()):
        print(f"  {task:30s}: {scores['string_match']:.2f}")


filtering__0.01__4096
  cwe                           : 93.33
  fwe                           : 83.33

filtering__0.01__4096  niah_multikey_1               : 100.00

  cwe                           : 93.33
  fwe                           : 83.33
  niah_multikey_1               : 100.00
  niah_multikey_2               : 100.00
  niah_multikey_3               : 100.00
  niah_multiquery               : 100.00
  niah_multivalue               : 100.00
  niah_multikey_2               : 100.00
  niah_single_1                 : 100.00
  niah_multikey_3               : 100.00
  niah_multiquery               : 100.00
  niah_multivalue               : 100.00
  niah_single_2                 : 100.00
  niah_single_1                 : 100.00
  niah_single_3                 : 100.00
  qa_1                          : 80.00
  qa_2                          : 62.50

filtering__0.01__8192
  cwe                           : 86.67
  fwe                           : 94.44
  niah_multikey_1               : 100

## 5. Save Results

In [11]:
import json

os.makedirs("results/vllm_ruler", exist_ok=True)

predictions_path = "results/vllm_ruler/predictions.csv"
df_all = pd.concat(all_predictions, ignore_index=True)
df_all.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_ruler/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/vllm_ruler/predictions.csv
Saved predictions to results/vllm_ruler/predictions.csv
Saved metrics to results/vllm_ruler/metrics.json
Saved metrics to results/vllm_ruler/metrics.json
